# Análisis hidrológico de la Cuenca del Río Suquía

Delimitación, red de drenaje, perfil del cauce principal y subcuencas a partir de un
DEM **Copernicus GLO-30**, usando [pysheds](https://github.com/mdbartos/pysheds).

**Requisitos:** `pip install pysheds rasterio numpy scipy matplotlib cartopy`
El archivo `cordoba_cop30_dem.tif` debe estar en la misma carpeta que esta notebook.

La notebook está dividida en secciones: (1) carga y acondicionamiento, (2) red de
drenaje, (3) perfil + métricas, (4) subcuencas, (5) nota sobre caudales.


## 0. Imports y parámetros


In [ ]:
import math
import numpy as np
from scipy import ndimage
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LightSource, LogNorm, ListedColormap
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pysheds.grid import Grid
import rasterio
import warnings
warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
FN  = "cordoba_cop30_dem.tif"
BBOX = (-65.0, -31.68, -63.6, -31.05)
DIRMAP = (64, 128, 1, 2, 4, 8, 16, 32)
OUTLET    = (-63.98, -31.38)     # salida de la cuenca (afueras E de Cordoba)
SEED_LAGO = (-64.45, -31.37)     # semilla sobre el embalse San Roque
AREA_PIXEL_KM2 = 0.0009          # nominal COP30 (~30 m)
Q_ESPECIFICO   = 0.0062          # m3/s por km2, calibrado al aforo de San Roque

# D8 -> desplazamiento (dfila, dcolumna)
OFFSETS = {64:(-1,0), 128:(-1,1), 1:(0,1), 2:(1,1), 4:(1,0), 8:(1,-1), 16:(0,-1), 32:(-1,-1)}


## 1. Carga y acondicionamiento del DEM

Suaviza levemente, rellena pits y depresiones, y **resuelve los flats con
`max_iter=1e9, eps=1e-12`** (clave: las superficies planas grandes como el embalse
no se resuelven con los parámetros por defecto y dejan celdas que "capturan" el flujo).
Luego calcula direcciones de flujo, acumulación, delimita la cuenca y detecta el lago.
Tarda ~1 minuto.


In [ ]:
with rasterio.open(FN) as src:
    nodata = src.nodata if src.nodata is not None else -9999

grid = Grid.from_raster(FN, nodata=nodata, window=BBOX)
dem  = grid.read_raster(FN, nodata=nodata, window=BBOX)
dem_raw = np.asarray(dem).astype(float)
aff = grid.affine
print("DEM:", dem.shape)

# Suavizado + acondicionamiento iterativo
sm = ndimage.gaussian_filter(dem, sigma=0.8)
conditioned = dem.copy(); conditioned[:] = sm
for i in range(5):
    conditioned = grid.fill_pits(conditioned)
    conditioned = grid.fill_depressions(conditioned)
    conditioned = grid.resolve_flats(conditioned, max_iter=int(1e9), eps=1e-12)
    if not grid.detect_pits(conditioned).any() and not grid.detect_flats(conditioned).any():
        break
print("Acondicionado en", i + 1, "pasadas")

# Ruteo y acumulacion
flowdir = grid.flowdir(conditioned, dirmap=DIRMAP)
fd  = np.asarray(flowdir)
acc = grid.accumulation(flowdir, dirmap=DIRMAP)
acc_arr = np.asarray(acc)

# Salida y cuenca
x_snap, y_snap = grid.snap_to_mask(acc > int(1800 / AREA_PIXEL_KM2), OUTLET)
catch = grid.catchment(x=x_snap, y=y_snap, fdir=flowdir, dirmap=DIRMAP, xytype="coordinate")
catch_arr = np.asarray(catch)

# Embalse San Roque: region plana conexa, sembrada sobre el lago
col0, row0 = grid.nearest_cell(*SEED_LAGO)
rango = ndimage.maximum_filter(dem_raw, size=3) - ndimage.minimum_filter(dem_raw, size=3)
labels, n = ndimage.label(rango < 1.0)
lab = int(labels[row0, col0])
if lab == 0:
    win = labels[max(0, row0-4):row0+5, max(0, col0-4):col0+5]; v = win[win > 0]
    lab = int(np.bincount(v).argmax()) if v.size else 0
lago = (labels == lab) if lab > 0 else np.zeros_like(catch_arr, bool)

# Area de celda (m2) por fila, corregida por latitud
rows = np.arange(dem_raw.shape[0])
lat_rows = aff.f + aff.e * (rows + 0.5)
area_fila = (abs(aff.a) * 111320 * np.cos(np.radians(lat_rows))) * (abs(aff.e) * 110540)

print("Cuenca delimitada. Lago:", int(lago.sum()), "celdas")


### Funciones auxiliares (área, red, perfil, caudal, dibujo)


In [ ]:
def haversine(lo1, la1, lo2, la2):
    R = 6371000.0; p1, p2 = math.radians(la1), math.radians(la2)
    dphi = math.radians(la2 - la1); dl = math.radians(lo2 - lo1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def lonlat(row, col):
    return aff.c + aff.a * (col + 0.5), aff.f + aff.e * (row + 0.5)

def area_km2(mask):
    return float((np.asarray(mask).sum(axis=1) * area_fila).sum() / 1e6)

def red(mask=None, umbral_km2=2.5):
    m = catch_arr if mask is None else np.asarray(mask)
    return (acc_arr > int(umbral_km2 / AREA_PIXEL_KM2)) & m & ~lago

def longitud_red_km(streams):
    rr, cc = np.where(streams); H, W = fd.shape; tot = 0.0
    for r, c in zip(rr, cc):
        off = OFFSETS.get(int(fd[r, c]))
        if off is None: continue
        nr, nc = r + off[0], c + off[1]
        if 0 <= nr < H and 0 <= nc < W:
            lo1, la1 = lonlat(r, c); lo2, la2 = lonlat(nr, nc)
            tot += haversine(lo1, la1, lo2, la2)
    return tot / 1000.0

def relieve(mask):
    v = dem_raw[np.asarray(mask)]; v = v[(np.isfinite(v)) & (v > -1000)]
    return float(v.max()), float(v.min())

def caudal_m3s(mask):
    return Q_ESPECIFICO * area_km2(mask)

def camino_principal():
    dist = np.asarray(grid.distance_to_outlet(x=x_snap, y=y_snap, fdir=flowdir,
                                              dirmap=DIRMAP, xytype="coordinate")).astype(float)
    dist[~catch_arr] = -1.0; dist[~np.isfinite(dist)] = -1.0
    r, c = np.unravel_index(int(np.argmax(dist)), dist.shape)
    ocol, orow = grid.nearest_cell(x_snap, y_snap); outlet = (orow, ocol)
    H, W = fd.shape; camino = [(r, c)]; visto = {(r, c)}
    for _ in range(H * W):
        if (r, c) == outlet: break
        off = OFFSETS.get(int(fd[r, c]))
        if off is None: break
        r, c = r + off[0], c + off[1]
        if not (0 <= r < H and 0 <= c < W) or (r, c) in visto: break
        visto.add((r, c)); camino.append((r, c))
    d = [0.0]; z = [dem_raw[camino[0]]]
    for i in range(1, len(camino)):
        lo1, la1 = lonlat(*camino[i-1]); lo2, la2 = lonlat(*camino[i])
        d.append(d[-1] + haversine(lo1, la1, lo2, la2)); z.append(dem_raw[camino[i]])
    z = np.array(z); z[z < -1000] = np.nan
    return np.array(d) / 1000.0, z

def encuadre(pad=0.08):
    ys, xs = np.where(catch_arr)
    ll = aff.c + aff.a * xs.min(); lr = aff.c + aff.a * (xs.max() + 1)
    lt = aff.f + aff.e * ys.min(); lb = aff.f + aff.e * (ys.max() + 1)
    return [ll - pad, lr + pad, lb - pad, lt + pad]

CIUDADES = {
    "Cordoba Cap.": (-64.188, -31.413, 0.02, -0.05),
    "La Calera":    (-64.335, -31.343, 0.02,  0.03),
    "Carlos Paz":   (-64.494, -31.424, -0.10, -0.05),
    "Cosquin":      (-64.466, -31.245, -0.12, -0.01),
    "La Falda":     (-64.489, -31.092, -0.11,  0.01),
}
def dibujar_ciudades(ax, ext, cuales=None):
    for nombre, (lon, lat, dx, dy) in CIUDADES.items():
        if cuales and nombre not in cuales: continue
        if ext[0] <= lon <= ext[1] and ext[2] <= lat <= ext[3]:
            ax.scatter(lon, lat, c="white", edgecolors="black", s=45, zorder=6, transform=ccrs.PlateCarree())
            ax.text(lon + dx, lat + dy, nombre, fontsize=10, fontweight="bold",
                    color="white", zorder=7, transform=ccrs.PlateCarree())

ls = LightSource(azdeg=315, altdeg=45)


## 2. Mapa de la red de drenaje

Criterio mostrado: celdas con área de acumulación > 2.5 km², dentro de la cuenca, sin el espejo del lago.


In [ ]:
umbral_red_km2 = 2.5; grosor = 2
streams = red(umbral_km2=umbral_red_km2)
red_acc = np.where(ndimage.binary_dilation(streams, iterations=grosor), acc_arr, np.nan)
shaded  = np.where(catch_arr, ls.hillshade(conditioned, fraction=1.0), np.nan)

ext = encuadre(); asp = (ext[1]-ext[0]) / (ext[3]-ext[2]); obj = 12.0
W, H = (obj, obj/asp) if asp >= 1 else (obj*asp, obj)
fig, ax = plt.subplots(figsize=(W, H), dpi=140, subplot_kw={'projection': ccrs.PlateCarree()})
ax.set_facecolor('#0d0826')
ax.imshow(shaded, cmap="gray", extent=grid.extent, alpha=0.35, interpolation="bilinear",
          transform=ccrs.PlateCarree(), zorder=1)
im = ax.imshow(red_acc, extent=grid.extent, cmap="autumn",
               norm=LogNorm(int(umbral_red_km2/AREA_PIXEL_KM2), acc_arr.max()),
               interpolation="none", transform=ccrs.PlateCarree(), zorder=2)
ax.imshow(np.where(lago, 1, np.nan), extent=grid.extent, cmap=ListedColormap(["#1f6feb"]),
          alpha=0.95, interpolation="none", transform=ccrs.PlateCarree(), zorder=3)
plt.colorbar(im, ax=ax, label="Area de acumulacion (celdas aguas arriba)", shrink=0.85, pad=0.02)
dibujar_ciudades(ax, ext)
ax.add_feature(cfeature.BORDERS, linestyle='--', alpha=0.5)
ax.gridlines(draw_labels=["bottom","left"], dms=True, x_inline=False, y_inline=False, alpha=0.2, color="white")
plt.title("Cuenca del Rio Suquia - Red de drenaje", fontsize=14, fontweight="bold", pad=15)
ax.set_extent(ext, crs=ccrs.PlateCarree())
plt.tight_layout(); plt.savefig("mapa_red.png", dpi=220, bbox_inches="tight", facecolor="white")
plt.show()


## 3. Perfil del cauce principal y métricas globales

El cauce principal es el **camino de flujo más largo**: del punto hidráulicamente más lejano hasta la salida, siguiendo las direcciones D8.


In [ ]:
dkm, zm = camino_principal()
L = float(dkm[-1]); z0 = float(np.nanmax(zm[:5])); zf = float(np.nanmin(zm[-5:]))
dz = z0 - zf; pend = dz / (L * 1000) * 100

streams = red(umbral_km2=2.5)
km_red = longitud_red_km(streams); area = area_km2(catch_arr)
zmax, zmin = relieve(catch_arr); Q = caudal_m3s(catch_arr)

print(f"Area total            : {area:,.0f} km2")
print(f"Longitud de red        : {km_red:,.0f} km")
print(f"Cota max / min         : {zmax:.0f} / {zmin:.0f} m  (relieve {zmax-zmin:.0f} m)")
print(f"Cauce principal        : {L:.1f} km | salto {dz:.0f} m | pendiente media {pend:.2f} %")
print(f"Caudal medio estimado  : ~{Q:.1f} m3/s")

fig, ax = plt.subplots(figsize=(12, 5), dpi=140)
base = float(np.nanmin(zm)) - 40
ax.fill_between(dkm, zm, base, color="#b08433", alpha=0.25)
ax.plot(dkm, zm, color="#1f6feb", lw=2.2)
ax.set_xlabel("Distancia desde la naciente mas lejana (km)")
ax.set_ylabel("Elevacion (m s.n.m.)")
ax.set_title("Perfil longitudinal del cauce principal - Rio Suquia", fontweight="bold")
ax.grid(alpha=0.3); ax.margins(x=0); ax.set_ylim(base, None)
txt = (f"Longitud : {L:.1f} km\nSalto    : {dz:.0f} m\nPendiente: {pend:.2f} %\n"
       f"Area     : {area:,.0f} km2\nCaudal   : ~{Q:.1f} m3/s")
ax.text(0.985, 0.95, txt, transform=ax.transAxes, ha="right", va="top", family="monospace",
        bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.92))
plt.tight_layout(); plt.savefig("perfil_suquia.png", dpi=220, bbox_inches="tight", facecolor="white")
plt.show()


## 4. Subcuencas

Las dos mayores (**San Antonio** y **Cosquín**) se delimitan por punto de cierre. El
**aporte al San Roque** se toma como la cuenca del punto de salida del lago (la celda
de mayor acumulación dentro del espejo, robusto y automático). Por diferencia salen
los **otros aportes al lago** (arroyos Las Mojarras / Los Chorrillos + drenaje directo)
y la **cuenca baja** aguas abajo del dique.


In [ ]:
umbral_snap = int(40 / AREA_PIXEL_KM2)   # engancha a canales > 40 km2
def cuenca_en(x, y):
    xs, ys = grid.snap_to_mask(acc > umbral_snap, (x, y))
    m = np.asarray(grid.catchment(x=xs, y=ys, fdir=flowdir, dirmap=DIRMAP, xytype="coordinate"))
    return m & catch_arr

san_antonio = cuenca_en(-64.503, -31.435)
cosquin     = cuenca_en(-64.462, -31.330)

# Aporte al San Roque = cuenca del punto de salida del lago
# (celda de mayor acumulacion dentro del espejo: por ahi sale toda el agua)
acc_lago = np.where(lago, acc_arr, -1.0)
r_out, c_out = np.unravel_index(int(acc_lago.argmax()), acc_lago.shape)
xo, yo = lonlat(r_out, c_out)
aporte = np.asarray(grid.catchment(x=xo, y=yo, fdir=flowdir, dirmap=DIRMAP, xytype="coordinate")) & catch_arr

sub = {
    "Rio San Antonio": san_antonio & ~cosquin,
    "Rio Cosquin":     cosquin & ~san_antonio,
    "Otros aportes al lago": aporte & ~(san_antonio | cosquin),
    "Cuenca baja (post dique)": catch_arr & ~aporte,
}
colores = {"Rio San Antonio":"#e4572e", "Rio Cosquin":"#2e86ab",
           "Otros aportes al lago":"#8ac926", "Cuenca baja (post dique)":"#9b5de5"}
orden = list(sub.keys())

print(f"{'Subcuenca':<28}{'Area km2':>10}{'Red km':>9}{'Relieve m':>11}{'Q m3/s':>9}")
print("-" * 67)
for k in orden:
    m = sub[k]; a = area_km2(m); km = longitud_red_km(red(mask=m))
    zx, zn = relieve(m); q = caudal_m3s(m)
    print(f"{k:<28}{a:>10,.0f}{km:>9,.0f}{zx-zn:>11.0f}{q:>9.1f}")
print("-" * 67)
print(f"Validacion aporte San Roque: {area_km2(aporte):,.0f} km2 (referencia INA ~1750 km2)")


In [ ]:
sub_id = np.full(catch_arr.shape, np.nan)
for i, k in enumerate(orden):
    sub_id[sub[k]] = i

shaded = np.where(catch_arr, ls.hillshade(conditioned, fraction=1.0), np.nan)
streams_disp = ndimage.binary_dilation(red(umbral_km2=4.0), iterations=1)

ext = encuadre(); asp = (ext[1]-ext[0]) / (ext[3]-ext[2]); obj = 12.0
W, H = (obj, obj/asp) if asp >= 1 else (obj*asp, obj)
fig, ax = plt.subplots(figsize=(W, H), dpi=140, subplot_kw={'projection': ccrs.PlateCarree()})
ax.set_facecolor('#0d0826')
ax.imshow(shaded, cmap="gray", extent=grid.extent, alpha=0.45, interpolation="bilinear",
          transform=ccrs.PlateCarree(), zorder=1)
ax.imshow(sub_id, extent=grid.extent, cmap=ListedColormap([colores[k] for k in orden]),
          vmin=-0.5, vmax=len(orden)-0.5, alpha=0.6, interpolation="none",
          transform=ccrs.PlateCarree(), zorder=2)
ax.imshow(np.where(streams_disp, 1, np.nan), extent=grid.extent, cmap=ListedColormap(["#10131f"]),
          alpha=0.55, interpolation="none", transform=ccrs.PlateCarree(), zorder=3)
ax.imshow(np.where(lago, 1, np.nan), extent=grid.extent, cmap=ListedColormap(["#1f6feb"]),
          alpha=0.95, interpolation="none", transform=ccrs.PlateCarree(), zorder=4)
dibujar_ciudades(ax, ext, cuales={"Cordoba Cap.","Carlos Paz","Cosquin","La Falda"})
handles = [mpatches.Patch(color=colores[k], label=f"{k}  ({area_km2(sub[k]):,.0f} km2)") for k in orden]
ax.legend(handles=handles, loc="lower left", fontsize=9, framealpha=0.92)
ax.gridlines(draw_labels=["bottom","left"], dms=True, x_inline=False, y_inline=False, alpha=0.2, color="white")
plt.title("Subcuencas del Rio Suquia (aporte al dique San Roque)", fontsize=14, fontweight="bold", pad=15)
ax.set_extent(ext, crs=ccrs.PlateCarree())
plt.tight_layout(); plt.savefig("mapa_subcuencas.png", dpi=220, bbox_inches="tight", facecolor="white")
plt.show()


## 5. Sobre el cálculo de caudales

Del DEM **no se puede sacar caudal directamente**: el modelo da área y pendiente, no
volumen de agua. La estimación de esta notebook usa un **caudal específico calibrado a
un aforo real**: el módulo del Suquía en San Roque es ~10,9 m³/s sobre ~1750 km² (INA),
o sea **q ≈ 0,0062 m³/s/km²** (escorrentía ~196 mm/año, coeficiente ~0,25 con lluvia
~800 mm/año). El caudal medio de cada cuenca se estima como **Q ≈ q · área**.

**Limitaciones importantes:**
- Es una estimación de orden de magnitud, no un balance hidrológico.
- El `q` específico está calibrado a la **cuenca serrana** (húmeda). Aplicarlo a la
  **cuenca baja** (llanura árida, con extracción de agua y regulación del dique)
  **sobreestima** su aporte. En la práctica, el caudal medio en la salida es similar al
  del aporte al San Roque (~10–11 m³/s), no la suma directa de todas las áreas.
- Un caudal confiable requiere series de precipitación/escorrentía y un modelo
  hidrológico (p. ej. método racional para crecidas, o balance mensual para módulo).

**Fuentes:** DEM Copernicus GLO-30 (ESA); caudal y subcuencas de referencia: INA / CIRSA.
